# Quantum AMC — Complete Experimental Notebook## Data-Efficient Quantum Machine Learning for Automatic Modulation Classification**WiCOMM-2026 · RadioML 2016.10a · 17 experiments**---### THE FINDINGThe quantum head was never the lever. The **6-dim bottleneck** — required only because a6-qubit circuit accepts exactly 6 inputs — costs **~0.124 accuracy** before any gate executes.Five classifier families with unlimited data all cap within **0.013** of each other on those6 features (0.517–0.530) while a raw-I/Q CNN reaches **0.744**.Every quantum-specific constraint was attacked (input aperture, exit aperture, non-convexity,Fourier structure). Best quantum config: **0.670**. Classical MLP on the same 6 numbers: **0.676**.Quantum converged to parity — never past it.---### HOW TO RUNSections 1–4 rebuild state (~10 min on T4 GPU). Sections 5+ are individual experiments — runwhichever you need. **Section 12 saves everything to Google Drive.****Runtime → Change runtime type → T4 GPU.** Quantum layers run on CPU regardless (PennyLanesimulates the statevector on CPU); the GPU accelerates CNN training and feature extraction.

## 1 — Setup

In [ ]:
!pip install -q pennylane pennylane-lightning gdownimport os, shutil, zipfile, pickle, time, json, glob, itertoolsimport numpy as np, torch, torch.nn as nn, pennylane as qmlimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_split, KFoldfrom sklearn.linear_model import LogisticRegression, RidgeClassifierfrom sklearn.svm import SVCfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.preprocessing import StandardScalerfrom scipy import stats as spsdevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)n_qubits, n_layers, SNR_MIN = 6, 4, 0print("device:", device, "| torch", torch.__version__, "| pennylane", qml.version())

## 2 — DatasetRadioML 2016.10a via Zenodo **DOI 10.5281/zenodo.18397070** (bit-identical to the original).Not DeepSig-direct (502/SSL errors) and not Kaggle. Licence **CC BY-NC-SA 4.0**.11 classes · SNR −20…+18 dB (2 dB steps) · ~1000 examples per (class,SNR) · 128 I/Q samples · 220,000 total.**No OFDM in this dataset** — a real coverage gap to state as a limitation.

In [ ]:
PKL = "RML2016.10a_dict.pkl"if not os.path.exists(PKL):    import urllib.request, tarfile    urllib.request.urlretrieve(        "https://zenodo.org/records/18397070/files/RML2016.10a.tar.bz2?download=1",        "RML2016.10a.tar.bz2")    with tarfile.open("RML2016.10a.tar.bz2","r:bz2") as t: t.extractall()    for r,_,fs in os.walk("."):        for f in fs:            if f.endswith("RML2016.10a_dict.pkl"): PKL = os.path.join(r,f)raw_d = pickle.load(open(PKL,"rb"), encoding="latin1")   # latin1 required in Python 3mods = sorted({k[0] for k in raw_d}); mod_to_idx = {m:i for i,m in enumerate(mods)}X,y,S = [],[],[]for (m,s),arr in raw_d.items():    X.append(arr); y += [mod_to_idx[m]]*arr.shape[0]; S += [s]*arr.shape[0]X = np.vstack(X).astype(np.float32); y = np.array(y); S = np.array(S)# per-sample L2 normalisation: model sees SHAPE, not loudnessX = X / (np.sqrt((X**2).sum(axis=(1,2), keepdims=True)) + 1e-8)del raw_dXtr,Xtmp,ytr,ytmp,Str,Stmp = train_test_split(X,y,S,test_size=0.4,random_state=SEED,stratify=y)Xva,Xte,yva,yte,Sva,Ste = train_test_split(Xtmp,ytmp,Stmp,test_size=0.5,random_state=SEED,stratify=ytmp)print("X:", X.shape, "| classes:", mods)print("split:", len(ytr), len(yva), len(yte))

## 3 — Models and helpers**The critical constraint:** `AngleEmbedding` accepts exactly `n_qubits` inputs. The CNN produces1024 features. So a 6-dim bottleneck is *required* — and that bottleneck is what costs the accuracy.

In [ ]:
class FeatureExtractor(nn.Module):    def __init__(self, d, L=128):        super().__init__()        self.features = nn.Sequential(            nn.Conv1d(2,32,3,padding=1), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.3),            nn.Conv1d(32,32,3,padding=1), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.3))        self.bottleneck = nn.Sequential(nn.Flatten(), nn.Linear(32*32,d), nn.Tanh())    def forward(self,x): return self.bottleneck(self.features(x))class ClassicalHead(nn.Module):    def __init__(self, d, n_classes, hidden=64, p=0.2):        super().__init__()        self.net = nn.Sequential(nn.Linear(d,hidden), nn.ReLU(), nn.Dropout(p),                                 nn.Linear(hidden,hidden), nn.ReLU(), nn.Dropout(p),                                 nn.Linear(hidden,n_classes))    def forward(self,x): return self.net(x)class LinearHead(nn.Module):    def __init__(self, d, n_classes):        super().__init__(); self.net = nn.Linear(d,n_classes)    def forward(self,x): return self.net(x)class RawCNN(nn.Module):    def __init__(self, n_classes, L=128):        super().__init__()        self.f = nn.Sequential(nn.Conv1d(2,64,3,padding=1), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.3),                               nn.Conv1d(64,64,3,padding=1), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.3))        self.c = nn.Sequential(nn.Flatten(), nn.Linear(64*32,128), nn.ReLU(),                               nn.Dropout(0.3), nn.Linear(128,n_classes))    def forward(self,x): return self.c(self.f(x))try:    qdev = qml.device("lightning.qubit", wires=n_qubits); QDIFF="adjoint"except: qdev = qml.device("default.qubit",  wires=n_qubits); QDIFF="backprop"@qml.qnode(qdev, interface="torch", diff_method=QDIFF)def qnode_ru(inputs, weights):    for l in range(n_layers):        qml.AngleEmbedding(inputs, wires=range(n_qubits))      # RE-UPLOAD each layer        qml.StronglyEntanglingLayers(weights[l:l+1], wires=range(n_qubits))    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]ru_shapes = {"weights": (n_layers, n_qubits, 3)}               # 4x6x3 = 72 paramsclass QuantumHeadRU(nn.Module):    def __init__(self, d, n_classes, n_layers):        super().__init__()        self.qlayer = qml.qnn.TorchLayer(qnode_ru, ru_shapes); self.out = nn.Linear(d,n_classes)    def forward(self,x): return self.out(self.qlayer(x*np.pi))  # (-1,1) -> (-pi,pi) anglesdef extract(ext, Xa, bs=1024):    outs=[]    with torch.no_grad():        for i in range(0,len(Xa),bs):            outs.append(ext(torch.tensor(Xa[i:i+bs]).to(device)).cpu().numpy())    return np.concatenate(outs)def knn_predict(Fk, yk, Fq, k=1):    from scipy.stats import mode    d2 = np.sum((Fq[:,None,:]-Fk[None,:,:])**2, axis=2)    return mode(yk[np.argsort(d2,axis=1)[:,:min(k,len(yk))]], axis=1, keepdims=False).modedef train_head(make_head, Fk, yk, Fq, yq, epochs=60, lr=1e-2):    head = make_head()    Xk = torch.tensor(Fk,dtype=torch.float32); Yk = torch.tensor(yk,dtype=torch.long)    opt = torch.optim.Adam(head.parameters(), lr=lr); crit = nn.CrossEntropyLoss(); head.train()    for _ in range(epochs): opt.zero_grad(); crit(head(Xk),Yk).backward(); opt.step()    head.eval()    with torch.no_grad():        return (head(torch.tensor(Fq,dtype=torch.float32)).argmax(1).numpy()==yq).mean()def full_data_mlp(Ftr_,ytr_,Fte_,yte_,n_classes,hidden=64,epochs=200,lr=1e-2):    net = nn.Sequential(nn.Linear(Ftr_.shape[1],hidden), nn.ReLU(),                        nn.Linear(hidden,hidden), nn.ReLU(), nn.Linear(hidden,n_classes)).to(device)    Xk = torch.tensor(Ftr_,dtype=torch.float32).to(device); Yk = torch.tensor(ytr_,dtype=torch.long).to(device)    o = torch.optim.Adam(net.parameters(),lr=lr); c = nn.CrossEntropyLoss()    for _ in range(epochs): o.zero_grad(); c(net(Xk),Yk).backward(); o.step()    net.eval()    with torch.no_grad():        return (net(torch.tensor(Fte_,dtype=torch.float32).to(device)).argmax(1).cpu().numpy()==yte_).mean()print("library ready | quantum:", qdev.name, QDIFF)

## 4 — Prototypical pretraining (removes the confound)**⚠ The confound that invalidated our first experiments:** the extractor was originally pretrainedwith a *linear probe on the same 11 classes* we then few-shot. Result: `full_probe_acc = 0.640` —a single linear layer already got 64%. The features weren't "useful", they were **already the answer**.**The fix:** prototypical (metric) pretraining on *base classes only*, then few-shot the held-out ones.Trains an embedding where same-class clusters, rather than a classifier that pre-solves specific classes.

In [ ]:
def sample_episode(Xp,yp,classes,n_way,k_shot,q_query,rng):    chosen = rng.choice(classes, size=n_way, replace=False)    sup,qry,remap = [],[],{c:i for i,c in enumerate(chosen)}    for c in chosen:        pick = rng.choice(np.where(yp==c)[0], size=k_shot+q_query, replace=False)        sup.append(pick[:k_shot]); qry.append(pick[k_shot:])    sup,qry = np.concatenate(sup), np.concatenate(qry)    return Xp[sup], np.array([remap[c] for c in yp[sup]]), Xp[qry], np.array([remap[c] for c in yp[qry]])def proto_loss(es,ys,eq,yq,n_way):    protos = torch.stack([es[ys==c].mean(0) for c in range(n_way)])    logits = -(torch.cdist(eq,protos)**2)          # negative squared distance    return nn.functional.cross_entropy(logits, torch.tensor(yq,device=eq.device)), logitsdef pretrain_prototypical(base_classes, episodes=2500, n_way=5, k_shot=5, q_query=15,                          lr=1e-3, seed=SEED, d=n_qubits, log_every=500):    tp = np.where((Str>=SNR_MIN) & (np.isin(ytr, base_classes)))[0]    Xp, yp = Xtr[tp], ytr[tp]    torch.manual_seed(seed)    ext = FeatureExtractor(d).to(device); opt = torch.optim.Adam(ext.parameters(), lr=lr)    rng = np.random.default_rng(seed); ext.train(); losses=[]    for ep in range(1, episodes+1):        Xs,ys,Xq_,yq_ = sample_episode(Xp,yp,base_classes,min(n_way,len(base_classes)),k_shot,q_query,rng)        es = ext(torch.tensor(Xs).to(device)); eq = ext(torch.tensor(Xq_).to(device))        loss,_ = proto_loss(es,ys,eq,yq_,min(n_way,len(base_classes)))        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())        if ep % log_every == 0: print(f"    ep {ep} | loss {np.mean(losses[-100:]):.3f}", flush=True)    ext.eval()    for p in ext.parameters(): p.requires_grad_(False)    return ext, float(np.mean(losses[-100:]))# THE CURATED HARD SET: QAM16+QAM64 are near-identical constellations (most confusable pair)NOVEL_HARD = ["QAM16","QAM64","8PSK","AM-DSB"]novel_c = sorted(mod_to_idx[m] for m in NOVEL_HARD)base_c  = sorted(i for i in range(len(mods)) if i not in novel_c)print("novel:", [mods[i] for i in novel_c]); print("base :", [mods[i] for i in base_c])ntr = np.where((Str>=SNR_MIN)&(np.isin(ytr,novel_c)))[0]nte = np.where((Ste>=SNR_MIN)&(np.isin(yte,novel_c)))[0]remap = {c:i for i,c in enumerate(novel_c)}Xtr_h, ytr_h = Xtr[ntr], np.array([remap[c] for c in ytr[ntr]])Xte_h, yte_h = Xte[nte], np.array([remap[c] for c in yte[nte]])print("\npretraining extractor on base classes ...")ext_hard, _ = pretrain_prototypical(base_c, episodes=2500, seed=SEED)Ftr_h, Fte_h = extract(ext_hard, Xtr_h), extract(ext_hard, Xte_h)rng0 = np.random.default_rng(7777); qi = rng0.choice(len(Fte_h), size=1000, replace=False)Fq_h, yq_h = Fte_h[qi], yte_h[qi]# NOVELTY CHECK — must be far below 0.640 (the pre-solved confound)nov = LogisticRegression(max_iter=500).fit(Ftr_h,ytr_h).score(Fte_h,yte_h)print(f"\n[novelty check] {nov:.3f}   (expected ~0.517; random 0.250; pre-solved confound was 0.640)")

## 5 — Experiment 1c: curated hard set (8 seeds, paired) ❌ NEGATIVE**Pre-committed rule:** quantum is real only if it beats classical **paired p<0.05 AND** shows a**negative K-slope** (Caro predicts the edge is *largest* at K=1 and shrinks).**Result:** negative at every K; slope **+0.00138** — the exact opposite of the data-efficiency signature.

In [ ]:
Ks, SEEDS = [1,5,10,20], [0,1,2,3,4,5,6,7]res_hard = {m:{K:[] for K in Ks} for m in ["classical_mlp","linear","quantum_ru","knn"]}paired_qc = {K:[] for K in Ks}t0=time.time()for seed in SEEDS:    rng = np.random.default_rng(seed)    for K in Ks:        idx = np.concatenate([rng.choice(np.where(ytr_h==c)[0], size=K, replace=False)                              for c in range(len(novel_c))])        Fk, yk = Ftr_h[idx], ytr_h[idx]           # SAME draw for every head (paired design)        torch.manual_seed(seed); a_mlp = train_head(lambda: ClassicalHead(n_qubits,len(novel_c)), Fk,yk,Fq_h,yq_h)        torch.manual_seed(seed); a_lin = train_head(lambda: LinearHead(n_qubits,len(novel_c)), Fk,yk,Fq_h,yq_h)        a_knn = (knn_predict(Fk,yk,Fq_h,k=1)==yq_h).mean()        torch.manual_seed(seed); a_q = train_head(lambda: QuantumHeadRU(n_qubits,len(novel_c),n_layers), Fk,yk,Fq_h,yq_h)        for m,a in [("classical_mlp",a_mlp),("linear",a_lin),("knn",a_knn),("quantum_ru",a_q)]:            res_hard[m][K].append(a)        paired_qc[K].append(a_q-a_mlp)        print(f"  seed {seed} K={K:2d} | q {a_q:.3f} vs mlp {a_mlp:.3f} (D={a_q-a_mlp:+.3f}) [{time.time()-t0:.0f}s]", flush=True)print("\n K | classical_mlp |    linear     |  quantum_ru   |     knn       | paired D  p")margins=[]for K in Ks:    ms = lambda m:(np.mean(res_hard[m][K]), np.std(res_hard[m][K]))    cm,cs=ms("classical_mlp"); lm,ls=ms("linear"); qm,qs=ms("quantum_ru"); km,ks=ms("knn")    d=np.array(paired_qc[K]); margins.append(d.mean())    _,p = sps.ttest_rel(res_hard["quantum_ru"][K], res_hard["classical_mlp"][K])    print(f"{K:2d} | {cm:.3f}+-{cs:.3f} | {lm:.3f}+-{ls:.3f} | {qm:.3f}+-{qs:.3f} | {km:.3f}+-{ks:.3f} | {d.mean():+.3f} p={p:.3f}")slope,*_ = sps.linregress(Ks, margins)print(f"\nK-shape slope {slope:+.5f} ->", "Caro-consistent" if slope<-0.001 else "ANTI data-efficiency")pickle.dump({"res_hard":res_hard,"paired_qc":paired_qc}, open("results_1c_hard.pkl","wb"))

## 6 — Experiment 1e: THE CEILING TEST ⭐ THE BREAKTHROUGH**The question:** is the ~0.49 plateau a *head* limit or an *embedding* limit?**The answer:** five completely different classifier families, given **all 24,001 training samples**,land within **0.013** of each other (0.517–0.530). A CNN on raw I/Q of the same classes hits **0.744**.Five algorithms agreeing that precisely means they're all correctly reading a representation thatalready threw the information away. **The head was never the lever.**

In [ ]:
ceiling = {}for nm, clf in [("linear (logreg)", LogisticRegression(max_iter=2000)),                ("RBF-SVM", SVC(kernel="rbf", C=10)),                ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)),                ("kNN (k=15)", KNeighborsClassifier(n_neighbors=15))]:    ceiling[nm] = clf.fit(Ftr_h,ytr_h).score(Fte_h,yte_h); print(f"  {nm:18s}: {ceiling[nm]:.3f}")ceiling["MLP (all data)"] = full_data_mlp(Ftr_h,ytr_h,Fte_h,yte_h,len(novel_c))print(f"  {'MLP (all data)':18s}: {ceiling['MLP (all data)']:.3f}")print("\ntraining raw-I/Q CNN (bypasses the bottleneck) ...")raw = RawCNN(len(novel_c)).to(device); opt = torch.optim.Adam(raw.parameters(), lr=1e-3)crit = nn.CrossEntropyLoss()ld = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(    torch.tensor(Xtr_h), torch.tensor(ytr_h,dtype=torch.long)), batch_size=256, shuffle=True)for ep in range(25):    raw.train()    for xb,yb in ld:        xb,yb = xb.to(device), yb.to(device)        opt.zero_grad(); crit(raw(xb),yb).backward(); opt.step()raw.eval(); preds=[]with torch.no_grad():    for i in range(0,len(Xte_h),1024):        preds.append(raw(torch.tensor(Xte_h[i:i+1024]).to(device)).argmax(1).cpu().numpy())acc_raw = (np.concatenate(preds)==yte_h).mean()ceiling["CNN on raw I/Q"] = acc_rawbest_embed = max(v for k,v in ceiling.items() if k!="CNN on raw I/Q")print(f"\n{'='*66}\nBest on 6-dim embedding (ALL data): {best_embed:.3f}")print(f"CNN on raw I/Q (no bottleneck):     {acc_raw:.3f}")print(f"=> BOTTLENECK COST: {acc_raw-best_embed:+.3f}")print("   Five classifier families within 0.013 of each other -> the HEAD is irrelevant.")print("   NOTE: use the Section-8 n=8 figure (+0.124) as the headline; a single")print("   extractor draw can be ~1.2 sigma off (see Part 5.10 of the record).")print("="*66)pickle.dump({"ceiling":ceiling,"raw_cnn":float(acc_raw)}, open("results_1e_ceiling.pkl","wb"))

## 7 — Vector 3: cumulant features ✅ BIG CLASSICAL WIN**Six hand-designed dimensions match thirty-two learned ones** (0.704 vs 0.701).**Why:** cumulants are *class-agnostic by construction* — never fit to the base classes, so **zerotransfer penalty**. The learned embedding must generalise from 7 seen classes to 4 unseen; cumulantsdon't have that problem.**⚠ But this advantage is TASK-SPECIFIC** — on the full 11-class supervised task cumulants *lose*to the CNN by 0.25 (see Section 11).

In [ ]:
def cumulant_features(Xa):    """(N,2,128) real I/Q -> (N,8) higher-order cumulant + envelope features."""    x = Xa[:,0,:] + 1j*Xa[:,1,:]    x = x / (np.sqrt(np.mean(np.abs(x)**2, axis=1, keepdims=True)) + 1e-12)    Mo = lambda p,q: np.mean(x**(p-q) * np.conj(x)**q, axis=1)    M20,M21 = Mo(2,0), Mo(2,1); M40,M41,M42 = Mo(4,0), Mo(4,1), Mo(4,2)    M60,M63 = Mo(6,0), Mo(6,3)    C20,C21 = M20, M21    C40 = M40 - 3*M20**2    C41 = M41 - 3*M20*M21    C42 = M42 - np.abs(M20)**2 - 2*M21**2    C60 = M60 - 15*M20*M40 + 30*M20**3    C63 = M63 - 9*C21*C42 - 6*C21**3    eps=1e-12; n2=np.abs(C21)+eps; amp=np.abs(x)    return np.stack([np.abs(C20)/n2, np.abs(C40)/n2**2, np.abs(C41)/n2**2, np.abs(C42)/n2**2,                     np.abs(C60)/n2**3, np.abs(C63)/n2**3,                     np.std(np.angle(x),axis=1),                     np.mean(amp**4,axis=1)/(np.mean(amp**2,axis=1)**2+eps)],                    axis=1).real.astype(np.float32)Ctr, Cte = cumulant_features(Xtr_h), cumulant_features(Xte_h)sc = StandardScaler().fit(Ctr); Ctr_s, Cte_s = sc.transform(Ctr), sc.transform(Cte)print(f"reference: 6-dim LEARNED = 0.620 (n=8) | raw CNN = {acc_raw:.3f} | random 0.250\n")for nm,clf in [("logreg",LogisticRegression(max_iter=3000)), ("RBF-SVM",SVC(kernel="rbf",C=10)),               ("RandomForest",RandomForestClassifier(n_estimators=300,random_state=SEED,n_jobs=-1))]:    print(f"  cumulants(8) + {nm:14s}: {clf.fit(Ctr_s,ytr_h).score(Cte_s,yte_h):.3f}")sel = [1,2,3,4,5,6]   # C40,C41,C42,C60,C63,phase-spread -> matched 6-dim aperturea6 = max(LogisticRegression(max_iter=3000).fit(Ctr_s[:,sel],ytr_h).score(Cte_s[:,sel],yte_h),         SVC(kernel="rbf",C=10).fit(Ctr_s[:,sel],ytr_h).score(Cte_s[:,sel],yte_h),         full_data_mlp(Ctr_s[:,sel],ytr_h,Cte_s[:,sel],yte_h,len(novel_c)))print(f"\n*** 6-dim CUMULANTS: {a6:.3f}  vs  6-dim LEARNED 0.620  -> {a6-0.620:+.3f} ***")Ctr6, Cte6 = np.tanh(Ctr_s[:,sel]).astype(np.float32), np.tanh(Cte_s[:,sel]).astype(np.float32)pickle.dump({"cum_6dim":float(a6)}, open("results_v3_cumulants.pkl","wb"))

## 8 — Vector 3 + Quantum: does quantum benefit from better inputs? ❌ NEGATIVEQuantum **did** improve (0.475 learned → 0.602 cumulants) — the circuit was genuinelyinformation-starved. **But it still loses, now significantly at EVERY K** (p ≤ 0.024).Better inputs made the deficit *more* detectable, because classical improved more.**And the K=1 instability replicated:** per-seed 0.533, 0.465, **0.282**, 0.531, **0.317** —two of five seeds collapsed near random (0.25). Variance 5× the classical heads'.

In [ ]:
Ks_c, SEEDS_C = [1,5,10,20], [0,1,2,3,4]rngq = np.random.default_rng(7777); qi2 = rngq.choice(len(Cte6), size=1000, replace=False)Fq_c, yq_c = Cte6[qi2], yte_h[qi2]res_c = {m:{K:[] for K in Ks_c} for m in ["classical_mlp","linear","quantum_ru","knn"]}for seed in SEEDS_C:    rng = np.random.default_rng(seed)    for K in Ks_c:        idx = np.concatenate([rng.choice(np.where(ytr_h==c)[0], size=K, replace=False)                              for c in range(len(novel_c))])        Fk, yk = Ctr6[idx], ytr_h[idx]        torch.manual_seed(seed); a_mlp = train_head(lambda: ClassicalHead(6,len(novel_c)), Fk,yk,Fq_c,yq_c)        torch.manual_seed(seed); a_lin = train_head(lambda: LinearHead(6,len(novel_c)), Fk,yk,Fq_c,yq_c)        a_knn = (knn_predict(Fk,yk,Fq_c,k=1)==yq_c).mean()        torch.manual_seed(seed); a_q = train_head(lambda: QuantumHeadRU(6,len(novel_c),n_layers), Fk,yk,Fq_c,yq_c)        for m,a in [("classical_mlp",a_mlp),("linear",a_lin),("knn",a_knn),("quantum_ru",a_q)]:            res_c[m][K].append(a)        print(f"  seed {seed} K={K:2d} | q {a_q:.3f} vs mlp {a_mlp:.3f} (D={a_q-a_mlp:+.3f})", flush=True)print("\n K | classical_mlp |    linear     |  quantum_ru   |     knn       | D(q-mlp)  p")margins=[]for K in Ks_c:    ms = lambda m:(np.mean(res_c[m][K]), np.std(res_c[m][K]))    cm,cs=ms("classical_mlp"); lm,ls=ms("linear"); qm,qs=ms("quantum_ru"); km,ks=ms("knn")    d = np.array(res_c["quantum_ru"][K])-np.array(res_c["classical_mlp"][K]); margins.append(d.mean())    _,p = sps.ttest_rel(res_c["quantum_ru"][K], res_c["classical_mlp"][K])    print(f"{K:2d} | {cm:.3f}+-{cs:.3f} | {lm:.3f}+-{ls:.3f} | {qm:.3f}+-{qs:.3f} | {km:.3f}+-{ks:.3f} | {d.mean():+.3f} p={p:.3f}")slope,*_ = sps.linregress(Ks_c, margins)print(f"\nK-shape slope {slope:+.5f} ->", "Caro-consistent" if slope<-0.001 else "ANTI data-efficiency")print(f"K=1 per-seed quantum: {[round(float(a),3) for a in res_c['quantum_ru'][1]]}")pickle.dump({"res_c":res_c}, open("results_v3_quantum.pkl","wb"))

## 9 — Vector 4: wide readout (27 observables) ⚖️ FIXES STABILITY, NOT ACCURACY**⚠ QCCN (Adv. Eng. Informatics 2026) independently proposed this** — "extended observablemeasurement strategy … multiple Hermitian observables". We *quantify* what it buys.**Result:** +0.040 accuracy on rich features, but a **5× variance collapse at K=1**(±0.106 → ±0.021, matching classical's ±0.020). Every catastrophic collapse rescued.**The readout is a real constraint for variance, secondary for mean.**⚠ Slow: adjoint does one backward pass *per observable*, so 27 obs ≈ 4.5× the cost of 6.

In [ ]:
Z_PAIRS = list(itertools.combinations(range(n_qubits),2))N_OBS = n_qubits + len(Z_PAIRS) + n_qubits          # 6 + 15 + 6 = 27@qml.qnode(qdev, interface="torch", diff_method=QDIFF)def qnode_wide(inputs, weights):    for l in range(n_layers):        qml.AngleEmbedding(inputs, wires=range(n_qubits))        qml.StronglyEntanglingLayers(weights[l:l+1], wires=range(n_qubits))    obs  = [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]    obs += [qml.expval(qml.PauliZ(i) @ qml.PauliZ(j)) for i,j in Z_PAIRS]   # entanglement-sensitive    obs += [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]             # coherence    return obsclass QuantumHeadWide(nn.Module):    def __init__(self, d_in, n_classes, n_layers):        super().__init__()        self.qlayer = qml.qnn.TorchLayer(qnode_wide, ru_shapes); self.out = nn.Linear(N_OBS, n_classes)    def forward(self,x): return self.out(self.qlayer(x*np.pi))res_v4 = {m:{K:[] for K in Ks_c} for m in ["quantum_narrow","quantum_wide","classical_mlp"]}for seed in SEEDS_C:    rng = np.random.default_rng(seed)    for K in Ks_c:        idx = np.concatenate([rng.choice(np.where(ytr_h==c)[0], size=K, replace=False)                              for c in range(len(novel_c))])        Fk, yk = Ctr6[idx], ytr_h[idx]        torch.manual_seed(seed); a_n = train_head(lambda: QuantumHeadRU(n_qubits,len(novel_c),n_layers), Fk,yk,Fq_c,yq_c)        torch.manual_seed(seed); a_w = train_head(lambda: QuantumHeadWide(n_qubits,len(novel_c),n_layers), Fk,yk,Fq_c,yq_c)        torch.manual_seed(seed); a_c = train_head(lambda: ClassicalHead(6,len(novel_c)), Fk,yk,Fq_c,yq_c)        res_v4["quantum_narrow"][K].append(a_n); res_v4["quantum_wide"][K].append(a_w)        res_v4["classical_mlp"][K].append(a_c)        print(f"  seed {seed} K={K:2d} | narrow {a_n:.3f} | wide {a_w:.3f} (D={a_w-a_n:+.3f}) | mlp {a_c:.3f}", flush=True)print(f"\n K | q_narrow(6) |  q_wide({N_OBS})  | classical   | D(wide-narrow)  p")for K in Ks_c:    ms = lambda m:(np.mean(res_v4[m][K]), np.std(res_v4[m][K]))    nm_,ns=ms("quantum_narrow"); wm,ws=ms("quantum_wide"); cm,cs=ms("classical_mlp")    d = np.array(res_v4["quantum_wide"][K])-np.array(res_v4["quantum_narrow"][K])    _,p = sps.ttest_rel(res_v4["quantum_wide"][K], res_v4["quantum_narrow"][K])    print(f"{K:2d} | {nm_:.3f}+-{ns:.3f} | {wm:.3f}+-{ws:.3f} | {cm:.3f}+-{cs:.3f} | {d.mean():+.3f} p={p:.3f}")print(f"\nK=1 STABILITY: narrow +-{np.std(res_v4['quantum_narrow'][1]):.3f} -> "      f"wide +-{np.std(res_v4['quantum_wide'][1]):.3f} (classical +-{np.std(res_v4['classical_mlp'][1]):.3f})")pickle.dump({"res_v4":res_v4}, open("results_v4_wide_readout.pkl","wb"))

## 10 — Vector 5: quantum kernel SVM ❌ CLEANEST NULLUses the **full 64-dim Hilbert space** for similarity — never collapses to expectation values.Convex training (global optimum guaranteed).**Result: QSVM ≈ RBF-SVM to three decimals at every K** (−0.001, −0.006, +0.003, −0.001).Not "quantum kernels underperform" — **quantum and classical Gaussian kernels compute functionallyidentical similarity structures.**Concentration check runs first: if off-diagonal std < 0.02 the Gram matrix is flat and results areuninformative. **Ours: mean 0.917, std 0.054 — structured.** This preempts the obvious objection.

In [ ]:
_kdev = qml.device("default.qubit", wires=n_qubits)@qml.qnode(_kdev)def _state(x):    qml.AngleEmbedding(x, wires=range(n_qubits), rotation='X')    for i in range(n_qubits): qml.CNOT(wires=[i,(i+1)%n_qubits])   # NOTE: qml.broadcast removed in new PennyLane    return qml.state()def states_of(Xa): return np.stack([np.asarray(_state(np.asarray(x,dtype=float))) for x in Xa])def qkernel(S1,S2): return np.abs(S1.conj() @ S2.T)**2             # ONE matmul, not NxM circuits_s = states_of(Ctr6[:8]); _K = qkernel(_s,_s); _off = _K[~np.eye(8,dtype=bool)]print(f"kernel diag ~{np.diag(_K).mean():.3f} | off-diag mean {_off.mean():.4f} std {_off.std():.4f}")print("CONCENTRATED - results uninformative" if _off.std()<0.02 else "kernel has structure - meaningful")S_test = states_of(Fq_c)res_q = {m:{K:[] for K in Ks_c} for m in ["qsvm","classical_mlp","rbf_svm","knn"]}for seed in SEEDS_C:    rng = np.random.default_rng(seed)    for K in Ks_c:        idx = np.concatenate([rng.choice(np.where(ytr_h==c)[0], size=K, replace=False)                              for c in range(len(novel_c))])        Fk, yk = Ctr6[idx], ytr_h[idx]        S_tr = states_of(Fk); Ktr, Kte = qkernel(S_tr,S_tr), qkernel(S_test,S_tr)        a_q = (SVC(kernel='precomputed',C=10).fit(Ktr,yk).predict(Kte)==yq_c).mean()        torch.manual_seed(seed); a_c = train_head(lambda: ClassicalHead(6,len(novel_c)), Fk,yk,Fq_c,yq_c)        a_r = ((SVC(kernel='rbf',C=10).fit(Fk,yk).predict(Fq_c)==yq_c).mean() if K>1               else (knn_predict(Fk,yk,Fq_c,k=1)==yq_c).mean())        a_n = (knn_predict(Fk,yk,Fq_c,k=1)==yq_c).mean()        for m,a in [("qsvm",a_q),("classical_mlp",a_c),("rbf_svm",a_r),("knn",a_n)]: res_q[m][K].append(a)        print(f"  seed {seed} K={K:2d} | qsvm {a_q:.3f} | mlp {a_c:.3f} | rbf {a_r:.3f} | knn {a_n:.3f}", flush=True)print("\n K |    qsvm     | classical_mlp |   rbf_svm   |     knn     | D(qsvm-mlp)  p")for K in Ks_c:    ms = lambda m:(np.mean(res_q[m][K]), np.std(res_q[m][K]))    qm,qs=ms("qsvm"); cm,cs=ms("classical_mlp"); rm,rs=ms("rbf_svm"); km,ks=ms("knn")    d = np.array(res_q["qsvm"][K])-np.array(res_q["classical_mlp"][K])    _,p = sps.ttest_rel(res_q["qsvm"][K], res_q["classical_mlp"][K])    print(f"{K:2d} | {qm:.3f}+-{qs:.3f} | {cm:.3f}+-{cs:.3f} | {rm:.3f}+-{rs:.3f} | {km:.3f}+-{ks:.3f} | {d.mean():+.3f} p={p:.3f}")print("\nSHARPER TEST - qsvm vs RBF (isolates QUANTUM geometry from kernel methods):")for K in Ks_c:    print(f"  K={K:2d}: {np.mean(res_q['qsvm'][K])-np.mean(res_q['rbf_svm'][K]):+.3f}")pickle.dump({"res_q":res_q}, open("results_v5_qkernel.pkl","wb"))

## 11 — Fusion (cumulants + CNN) ❌ DEAD END**Both attempts landed BELOW the CNN alone** (v1 0.4908, v2 0.4620 vs CNN 0.5270). A workingcombiner can always copy the CNN logits, so under-performing means it isn't learning.**But the deeper reason: there was nothing to fuse.** Cumulants are *strictly dominated* by the CNNfrom −6 to +2 dB (by 0.20–0.28) and merely tied above +4 dB. A combiner only helps when componentshave **different** strengths.**⚠ This falsified our prediction** — cumulants beat the CNN at 0 dB on the *4-class novel* task(0.534 vs 0.520) but lose by 0.247 on the *11-class supervised* task.**The cumulant advantage is task-specific: it comes from class-agnosticism on unseen classes.**

In [ ]:
RUN_FUSION = False   # set True to reproduce the dead end (~20 min)if RUN_FUSION:    Ctr_all, Cte_all = cumulant_features(Xtr), cumulant_features(Xte)    sc_all = StandardScaler().fit(Ctr_all)    Ctr_all, Cte_all = sc_all.transform(Ctr_all), sc_all.transform(Cte_all)    cnn = RawCNN(len(mods)).to(device); opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)    crit = nn.CrossEntropyLoss()    ld = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(        torch.tensor(Xtr), torch.tensor(ytr,dtype=torch.long)), batch_size=256, shuffle=True)    for ep in range(30):        cnn.train()        for xb,yb in ld:            xb,yb = xb.to(device), yb.to(device)            opt.zero_grad(); crit(cnn(xb),yb).backward(); opt.step()    cnn.eval()    def cnn_logits(Xa,bs=1024):        outs=[]        with torch.no_grad():            for i in range(0,len(Xa),bs): outs.append(cnn(torch.tensor(Xa[i:i+bs]).to(device)).cpu().numpy())        return np.concatenate(outs)    Ltr,Lte = cnn_logits(Xtr), cnn_logits(Xte)    acc_cnn = (Lte.argmax(1)==yte).mean()    clf_c = RandomForestClassifier(n_estimators=400,random_state=SEED,n_jobs=-1).fit(Ctr_all,ytr)    Pc_tr,Pc_te = clf_c.predict_proba(Ctr_all), clf_c.predict_proba(Cte_all)    acc_cum = (Pc_te.argmax(1)==yte).mean()    fuse = full_data_mlp(np.hstack([Ltr,Pc_tr,Ctr_all]), ytr,                         np.hstack([Lte,Pc_te,Cte_all]), yte, len(mods), hidden=128, epochs=300)    print(f"CNN {acc_cnn:.4f} | cumulants {acc_cum:.4f} | FUSION {fuse:.4f}")else:    print("Fusion skipped. Recorded results:")    print("  CNN on raw I/Q : 0.5270")    print("  cumulants + RF : 0.4557")    print("  FUSION v1      : 0.4908  (below CNN)")    print("  FUSION v2      : 0.4620  (worse - OOF logits weaker than test logits)")    print("  => features are NOT complementary; do not pursue")

## 12 — SAVE EVERYTHING TO GOOGLE DRIVESaves all result pickles, trained models, features, the notebook, and a manifest to`MyDrive/QAMC_project/` with a timestamped run folder.

In [ ]:
from google.colab import drivedrive.mount('/content/drive', force_remount=True)STAMP = time.strftime("%Y%m%d_%H%M")DEST  = f"/content/drive/MyDrive/QAMC_project/run_{STAMP}"os.makedirs(DEST, exist_ok=True)os.makedirs(f"{DEST}/results", exist_ok=True)os.makedirs(f"{DEST}/models",  exist_ok=True)os.makedirs(f"{DEST}/features",exist_ok=True)# 1. modelsfor nm, obj in [("ext_hard", ext_hard), ("raw_cnn", raw)]:    try: torch.save(obj.state_dict(), f"{DEST}/models/model_{nm}.pt"); print("saved model:", nm)    except Exception as e: print("skip", nm, e)# 2. features (expensive to regenerate)try:    np.savez_compressed(f"{DEST}/features/features_hard.npz",        Ftr_h=Ftr_h, ytr_h=ytr_h, Fte_h=Fte_h, yte_h=yte_h, Fq_h=Fq_h, yq_h=yq_h,        Ctr6=Ctr6, Cte6=Cte6)    print("saved features")except Exception as e: print("features skip:", e)# 3. every result pickle in cwdfor f in glob.glob("*.pkl"):    shutil.copy(f, f"{DEST}/results/{f}"); print("saved result:", f)# 4. master bundle of in-memory resultslive = {}for nm in ["res_hard","paired_qc","ceiling","res_c","res_v4","res_q","res_v1","res_fz","res_w"]:    if nm in globals(): live[nm] = globals()[nm]meta = {"timestamp": STAMP, "mods": mods, "NOVEL_HARD": NOVEL_HARD,        "base_c": [int(i) for i in base_c], "novel_c": [int(i) for i in novel_c],        "n_qubits": n_qubits, "n_layers": n_layers,        "acc_raw_cnn": float(acc_raw) if 'acc_raw' in globals() else None,        "novelty_hard": float(nov) if 'nov' in globals() else None}pickle.dump({"live":live,"meta":meta}, open(f"{DEST}/results/ALL_RESULTS_master.pkl","wb"))json.dump(meta, open(f"{DEST}/manifest.json","w"), indent=2, default=str)# 5. the notebook itselffor nb in glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb") + glob.glob("*.ipynb"):    try: shutil.copy(nb, f"{DEST}/{os.path.basename(nb)}"); print("saved notebook:", os.path.basename(nb))    except Exception: passprint(f"\n{'='*60}\nSAVED TO: {DEST}")for root,_,fs in os.walk(DEST):    for f in fs: print("  ", os.path.relpath(os.path.join(root,f), DEST))print("="*60)

---# NEXT STEPS## ⭐ Frontier 3 — The Fragility Paradox (do this first, ~20 min)Our re-uploading circuit amplifies input perturbations by up to **~12.6×** (Lipschitz ≈ 4π fromL=4 nested sinusoids). Terrible for classification under channel noise — but **amplification isexactly what a detector wants**.**Test:** measure ‖Δoutput‖/‖δ‖ for quantum vs classical to verify the 12.6× prediction, then run aclean-vs-subtly-perturbed detection task and compare **ROC-AUC**.**This is the ONLY remaining direction our mechanism predicts could WIN.** A positive result changesthe paper's ending from *"quantum doesn't help at classification"* to *"quantum doesn't help atclassification, but its failure mode makes it a superior detector."*## Frontier 1 — Aperture Optimiser (~1 hr)We have two points on the "what 6 numbers?" axis: learned (0.620) and cumulants (0.704). Nobody hasasked what the **optimum** is. Tractable version: assemble ~40 cheap candidate features, search forthe best 6-subset by cross-validated accuracy → an empirical **ceiling on the 6-dim aperture**.*(Avoid direct mutual-information maximisation — MI estimation in 6 continuous dims is unstable.)*## ⛔ DO NOT RE-RUN- **Frontier 4 (sequential multiplexing)** = Vector 1, already run and failed (0.463 vs 0.676)- **Vector 2 (dual-stream)** — design cannot produce an interpretable negative- **Fusion** — features are not complementary- **Frontier 2 (quantum-first)** — 256-dim angle embedding needs 256 qubits; not simulable---# PAPER FRAMING> **"The feature bottleneck, not the classifier, determines automatic modulation classification> performance: a controlled study of hybrid quantum–classical models"**Lead with the **architectural** finding, not the quantum one — it reaches readers who don't careabout QML, and it's what the evidence supports most strongly.**Related work:** QCCN (Adv. Eng. Inf. 2026) and QTL (arXiv:2510.16301) both show the classicalfront-end matters, but **neither compares the quantum head against a matched classical head onidentical frozen features.** We supply that control.**Limitations:** 6 qubits · exact simulation only (which *strengthens* the negative) · single dataset,no OFDM · our 0.527 backbone is weaker than published ResNet/transformer SOTA (0.62–0.67) · wemeasured channel-noise robustness, not **adversarial** robustness — the negative does not extend there.